In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df=pd.read_csv(r"C:\Users\xhu70\Projects\twel_data_collection\data\sensor_data.csv")

In [ ]:
df.head()

In [ ]:
df["timestamp"]=pd.to_datetime(df["timestamp"])
df = df.dropna()
df = df.reset_index(drop=True)

In [ ]:
a, b = 17.27, 237.7
alpha = (a * df['temperature_c']) / (b + df['temperature_c']) + np.log(df['humidity_pct'] / 100)
df['dewpoint'] = (b * alpha) / (a - alpha)
df["differences"]=df["temperature_c"]-df['dewpoint']


In [ ]:
df["hour_sin"]=np.sin(2*np.pi*df["timestamp"].dt.hour/24)
df["hour_cos"]=np.cos(2*np.pi*df["timestamp"].dt.hour/24)
df["pressure_tendency"] = df["pressure_hpa"].diff(180)
df["humidity_tendency"] = (df["dewpoint"].diff(180))
df["temperature_tendency"]=df["temperature_c"].diff(360)
print(df["humidity_tendency"])

In [ ]:
features=["temperature_c","dewpoint","pressure_tendency"]

In [ ]:
HORIZON=180
LOOKBACK=180
TARGET="temperature_c"

In [ ]:
def make_dataset(df,horizon,lookback,target,features):

    X,y,y_current=[],[],[]

    data=df[features]
    tgt=df[target]
    
    for i in range(lookback, len(df)-horizon):
        X.append(data[i-lookback:i].values.flatten())
        y.append(tgt[i+horizon])
        y_current.append(tgt[i])
    
    return X,y,y_current

In [ ]:
X,y,y_naive= make_dataset(df,HORIZON,LOOKBACK,TARGET,features)


In [ ]:
split_train = int(0.6 * len(X))
split_val   = int(0.8 * len(X))

X_train = X[:split_train]
y_train = y[:split_train]

X_val   = X[split_train:split_val]
y_val   = y[split_train:split_val]

X_test  = X[split_val:]
y_test  = y[split_val:]


In [ ]:
import xgboost as xgb

In [ ]:
model=xgb.XGBRegressor(n_estimators=1000, learning_rate=0.01, max_depth=3,early_stopping_rounds=20,random_state=42)
model.fit(X_train,y_train,eval_set=[(X_val,y_val)])

In [ ]:
y_pred=model.predict(X_test)

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
mask = np.array(y_test) < 500
rmse1 = np.sqrt(mean_squared_error(
    np.array(y_pred)[mask],
    np.array(y_test)[mask]
))



In [ ]:
# y_naive是当前温度，y_naive[i-1]是上一步
y_naive_arr = np.array(y_naive)

# 取测试集的当前温度和上一步温度
current_temp = y_naive_arr[split_val:]      # t时刻温度
prev_temp    = y_naive_arr[split_val-1:-1]  # t-1时刻温度

slope = current_temp - prev_temp
naive_linear = current_temp + slope * HORIZON  # 外推6步

mask = np.array(y_test) < 500
rmse_linear = np.sqrt(mean_squared_error(
    naive_linear[mask],
    np.array(y_test)[mask]
))

print(f"Linear Extrapolation baseline RMSE: {rmse_linear:.4f}°C")
print(f"XGBoost RMSE:                       {rmse1:.4f}°C")
print(f"提升: {(rmse_linear - rmse1) / rmse_linear * 100:.1f}%")

In [ ]:
plt.figure(figsize=(14, 4))
plt.plot(y_test, label="真实值")
plt.plot(y_pred, label="预测值")
plt.legend()
plt.show()

In [ ]:
feature_names = []
for t in range(LOOKBACK, 0, -1):
    for f in features:
        feature_names.append(f"{f}_t-{t}")

model.get_booster().feature_names = feature_names
xgb.plot_importance(model, importance_type="gain", max_num_features=10)
plt.show()